# Rubrik adapter'ı — eğrinin ikinci noktası, kurtarılan checkpoint'ten

`rubric-curve` eğitimi 12 saatlik oturum duvarına çarptı: 200 adımın ~197'sinde
`exit 137`, yani `adapter_model.safetensors` hiç yazılmadı. Ama Kaggle o
koşunun `/kaggle/working`'ini yine de yayımladı (178,95 MB) ve `--save-steps 25`
sigortası ödedi — son iki Trainer checkpoint'i duruyor.

Bu notebook o checkpoint'i ölçüyor. **Eğitimi tekrarlamıyor.** Tekrarlamanın
maliyeti ~7,7 saat; bunun maliyeti ~2 saat, ve ölçtüğü nokta eğri için daha
iyisi: 800 değil ~1400 satır geçişi, yani tam koşunun (4800) yaklaşık %29'u.

**Neden 800 değil 1400.** `machine_shape: NvidiaTeslaT4` iki kart veriyor ve HF
Trainer üzerine sorulmadan DataParallel koyuyor: `--batch-size 1 --grad-accum 4`
adım başına 4 değil **8** satır demek. Eğitim notebook'unun bütçesi
`train_qlora_qwen.py`'nin `world_size` çarpanı olmayan çıktısından yazılmıştı,
o yüzden 200 adım 800 değil 1600 satır geçişi etti ve duvara çarptı.

Bu yüzden aşağıda satır geçişi **tahmin edilmiyor**: checkpoint'in kendi
`trainer_state.json`'ındaki `epoch` alanından okunuyor. Eğitim setinde 1600
satır var, dolayısıyla `epoch × 1600` gerçekte kaç satır görüldüğünün kaydı —
ve dosya adı da o sayıdan üretiliyor, elle yazılmıyor.

## Hangi sayıya bakılacak

`absent_rate` değil — taban onu zaten %89 yapıyor ve pilot yerinde saydığını
gösterdi (89,9 → 90,9). Bir metriğin tavanı adapter'ın kazancını değil, tabanın
yeterliliğini ölçer; Flutter v8 tam olarak burada yanıldı. Karar verecek olanlar
`present_score_mae` (düşük iyi) ve `hallucinated_quotes`.

Taban **aynı oturumda** ölçülüyor. Eğitim başka oturumda kaldı, o bir sorun
değil: farkın anlamlı olması için taban ile adapter'ın aynı kütüphane
sürümleriyle ve aynı greedy çözmeyle koşması yeterli, ikisi de burada oluyor.

Contrast yok — eğitim notebook'unun kararı korunuyor. Bu koşunun sorusu "kuralı
mı öğrendi bankayı mı" değil, "daha fazla eğitim ne veriyor". Contrast tam
koşunun ölçümüne ait ve ~40 dakika daha yerdi.

In [ ]:
import glob, json, os, shutil, subprocess, sys
import torch

assert torch.cuda.is_available(), "GPU acik degil - Settings > Accelerator > GPU T4"
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "sm_%d%d" % cap)
print("bellek: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

# Fail here, in five seconds, rather than after an 8 GB download. A P100 is
# sm_60: Kaggle's torch build does not support it at all, and bitsandbytes needs
# sm_75 for 4-bit NF4. The Flutter run landed on one because kernel-metadata
# omitted machine_shape, and the error arrived half an hour in wearing a
# different mask.
assert cap >= (7, 5), (
    f"sm_{cap[0]}{cap[1]} yetersiz - 4-bit NF4 icin T4 (sm_75) gerekiyor. "
    "Settings > Accelerator > GPU T4 x2")

In [ ]:
# Qwen3 icin transformers >= 4.51 gerekiyor; Kaggle imaji eskiyse sessizce
# 'unknown architecture' ile duser.
!pip -q install -U "transformers>=4.51" "peft>=0.11" "bitsandbytes>=0.43" "accelerate>=0.30" datasets 2>&1 | tail -2
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__, "| bnb", bitsandbytes.__version__)
# torchao kaldiriliyor, yukseltilmiyor. peft'in LoRA dispatcher'i sardigi her
# kuantize OLMAYAN Linear icin is_torchao_available() soruyor ve o fonksiyon
# uyumsuz surumde False donmek yerine ImportError firlatiyor. Kaggle imaji
# 0.10.0 tasiyor, peft ('peft>=0.11' artik 0.20'ye cozuluyor) >0.16.0 istiyor.
#
# Tuzak fp16 kolunda: 4-bit'te bitsandbytes kendi Linear4bit'ini once
# eslestirdigi icin dispatcher'a hic varilmiyor. colab-pilot-eval bunu bir kez
# odedi ve cozdu; buraya tasinmadigi icin rubric-curve-eval ayni duvara carpti
# — taban olcumu bittikten sonra, adapter gecisinin ilk saniyesinde.
#
# Silmek find_spec'i None yapar ve kontrol False doner, ki dogru cevap odur:
# torchao nicemlemesi kullanmiyoruz. Yukseltmek torch'u da suruklerdi.
!pip -q uninstall -y torchao 2>&1 | tail -1


In [ ]:
def find_mount(slug, marker):
    """Locate one input mount by the dataset/kernel slug in its path.

    Not by filename. Two mounts are attached here and both came out of the same
    training run, so a filename can appear in either; the slug is the only thing
    in the path that tells them apart. The rule is inherited from rubric-eval,
    where a kernel_sources mount carries the whole of a previous /kaggle/working
    — data files and scripts included — and matching on a data file picks a mount
    by luck.

    Recursive on top of that, because the mount depth is not a promise: the same
    dataset has appeared directly under /kaggle/input and, on the next run, one
    level deeper under /kaggle/input/datasets.
    """
    hits = [p for p in glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
            if slug.split("/")[-1] in p]
    assert hits, (f"'{slug}' bagli degil (aranan: {marker}). "
                  f"Kaggle > Notebook > Add Input, ve surumun islenmesi bitmis olmali.")
    return os.path.dirname(sorted(hits, key=len)[0])


for root, dirs, files in os.walk("/kaggle/input"):
    print(root, "->", sorted(files)[:4], "..." if len(files) > 4 else "")
    if root.count("/") > 6:
        dirs.clear()

In [ ]:
WORK = "/kaggle/working"
DATA = find_mount("emrahik/rubric-dataset", "rubric_eval.jsonl")
print("veri seti:", DATA)

os.makedirs(f"{WORK}/data", exist_ok=True)
for f in os.listdir(DATA):
    dst = f"{WORK}/data/{f}" if f.endswith(".jsonl") else f"{WORK}/{f}"
    shutil.copy(f"{DATA}/{f}", dst)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

## Adapter nereden geliyor

`kernel_sources` **değil**, dataset. Denendi ve Kaggle reddetti: iptal edilmiş
bir sürüm geçerli kernel kaynağı sayılmıyor —

```
The following are not valid kernel sources and could not be added to the
kernel: ['emrahik/rubric-curve']
```

ve daha kötüsü, push bu hatayla **başarılı** dönüyor; kernel girdisiz kalıyor.
`rubric-eval` bu yolu kullanabiliyor çünkü onun kaynağı temiz biten bir koşu.
Duvara çarpan koşunun çıktısı yalnızca `kaggle kernels output` ile indirilebilir
oluyor, o yüzden `checkpoint-175`'in çıkarım dosyaları
`emrahik/rubric-curve-adapter` olarak ayrıca yüklendi — `colab-pilot-adapter`
ile aynı kalıp. `optimizer.pt` dışarıda: 55 MB Adam durumu, buradan devam eden
bir şey yok.

Satır geçişi yine de **elle yazılmıyor**, checkpoint'in kendi
`trainer_state.json`'ından okunuyor.

In [ ]:
ADAPTER = find_mount("emrahik/rubric-curve-adapter", "adapter_model.safetensors")
print("adapter:", ADAPTER, sorted(os.listdir(ADAPTER)))

# The row count is read, not computed. epoch is the fraction of the 1600-row
# training set the optimizer actually consumed; multiplying it out is the only
# statement about this run's position on the curve that cannot be off by the
# world_size factor that killed the training kernel's budget.
st = json.load(open(f"{ADAPTER}/trainer_state.json"))
step, epoch = st["global_step"], st["epoch"]
ROWS = round(epoch * 1600)
print(f"\nglobal_step {step}  epoch {epoch:.3f}  ->  {ROWS} satir gecisi "
      f"({ROWS / 4800:.0%} of tam kosu)")
print(f"kontrol: {step} adim x 8 satir/adim = {step * 8}")

losses = [h for h in st["log_history"] if "loss" in h]
print(f"train loss: {losses[0]['loss']:.3f} (adim {losses[0]['step']})"
      f"  ->  {losses[-1]['loss']:.3f} (adim {losses[-1]['step']})")

## Ölçüm — taban ve adapter, tek koşuda

60 held-out satır, `rubric-eval`'in üretim ayarıyla aynı. Script önce tabanı,
sonra adapter'ı ölçer ve deltayı basar. İki geçiş yaklaşık 1,6 saat; oturum
duvarına 10 saatten fazla mesafe var, yani bu koşu gözetimsiz bırakılabilir.

In [ ]:
OUT = f"out/curve_{ROWS}.json"

# subprocess.run, `!` degil. `!`'in cikis kodu hicbir yere gitmez: rubric-train'in
# ilk kosusunda egitim adim 0'da CUDA OOM ile oldu, hucre devam etti, ve Kaggle
# kernel'i COMPLETE kaydetti — geriye kanit olarak yalnizca bos bir dizin kaldi.
r = subprocess.run([sys.executable, "rubric_eval.py",
                    "--data", "data/rubric_eval.jsonl",
                    "--base-model", "Qwen/Qwen3-4B-Instruct-2507",
                    "--adapter", ADAPTER,
                    "--limit", "60",
                    "--out", OUT])
assert r.returncode == 0, f"olcum coktu (exit {r.returncode}) — log yukarida"

In [ ]:
res = json.load(open(OUT))
print(json.dumps(res, indent=2, ensure_ascii=False))

b, a = res["base"], res["adapter"]
print(f"\n{ROWS} satir gecisi, uretim karisimi ve uretim eval seti\n")
print(f"{'olcum':<22}{'taban':>10}{'adapter':>10}{'delta':>10}")
for k in ("schema_valid", "completed", "absent_rate",
          "present_score_mae", "hallucinated_quotes"):
    bv, av = b.get(k), a.get(k)
    fmt = lambda v: "-" if v is None else f"{v:.3f}"
    d = "-" if (bv is None or av is None) else f"{av - bv:+.3f}"
    print(f"{k:<22}{fmt(bv):>10}{fmt(av):>10}{d:>10}")

# Bir sonraki karar bu iki satirdan cikiyor. Kazanc pilotun gordugu mertebedeyse
# egri erken duzlesiyor demektir ve tam kosunun 38 saati gerekcesiz kalir.
print("\nKarar metrikleri: present_score_mae ve hallucinated_quotes.")
print("absent_rate tavanda — tabanin yeterliligini olcer, adapter'in kazancini degil.")
print(f"\nPilot: %1.75 egitim, MAE 0.813 -> 0.631, hallucinated 3.25% -> 1.82%")
print(f"Bu kosu: {ROWS / 4800:.0%} egitim. Kazanc buna oranli mi, yoksa duzlesti mi?")